In [4]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Perceptron
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

# ==========================================
# 1. TẠO VÀ CHUẨN BỊ TẬP DỮ LIỆU (DATASET)
# ==========================================
# Mô phỏng tập dữ liệu giao dịch gian lận (Phân lớp nhị phân)
# 0: Giao dịch bình thường (95%), 1: Giao dịch gian lận (5%)
X_raw, y_raw = make_classification(
    n_samples=5000, 
    n_features=10, 
    n_informative=8, 
    n_redundant=2, 
    weights=[0.95, 0.05], # Dữ liệu lệch lớp thực tế
    random_state=42
)

# Chuyển thành DataFrame để quản lý dữ liệu
feature_names = [f"Feature_{i+1}" for i in range(10)]
df = pd.DataFrame(X_raw, columns=feature_names)
df['Is_Fraud'] = y_raw

print("--- 5 dòng dữ liệu đầu tiên ---")
print(df.head())
print("\nPhân bố nhãn (0: Bình thường, 1: Gian lận):")
print(df['Is_Fraud'].value_counts())

# ==========================================
# 2. CHIA TẬP DỮ LIỆU TRAIN / TEST
# ==========================================
X = df.drop(columns=['Is_Fraud'])
y = df['Is_Fraud']

# Chia tỷ lệ 80% Train - 20% Test, sử dụng stratify để giữ nguyên tỷ lệ nhãn
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ==========================================
# 3. TIỀN XỬ LÝ DỮ LIỆU (FEATURE SCALING)
# ==========================================
# Perceptron rất nhạy cảm với thang đo của đặc trưng, cần dùng StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ==========================================
# 4. XÂY DỰNG VÀ HUẤN LUYỆN MÔ HÌNH PERCEPTRON
# ==========================================
# Tối ưu siêu tham số Perceptron: 
# - class_weight='balanced': Giúp xử lý dữ liệu lệch lớp (quan trọng với gian lận)
# - penalty='l2' hoặc 'elasticnet': Thêm chính quy hóa để tránh overfitting
# - max_iter & eta0: Tốc độ học và số vòng lặp tối đa
model = Perceptron(
    max_iter=1000, 
    eta0=0.01, 
    random_state=42, 
    class_weight='balanced', 
    penalty='l2'
)

# Huấn luyện mô hình
model.fit(X_train_scaled, y_train)

# ==========================================
# 5. DỰ BÁO VÀ ĐÁNH GIÁ MÔ HÌNH
# ==========================================
y_pred = model.predict(X_test_scaled)

# Tính toán các độ đo yêu cầu
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("\n================ DỰ BÁO KẾT QUẢ ================")
print(f"Accuracy  (Độ chính xác tổng quan): {acc:.4f}")
print(f"Precision (Độ chính xác lớp Fraud): {prec:.4f}")
print(f"Recall    (Độ nhạy bắt Fraud)     : {rec:.4f}")
print(f"F1-score  (Độ đo dung hòa F1)     : {f1:.4f}")
print("\nMa trận nhầm lẫn (Confusion Matrix):")
print(confusion_matrix(y_test, y_pred))
print("\nBáo cáo chi tiết (Classification Report):")
print(classification_report(y_test, y_pred, target_names=['Bình thường (0)', 'Gian lận (1)']))

--- 5 dòng dữ liệu đầu tiên ---
   Feature_1  Feature_2  Feature_3  Feature_4  Feature_5  Feature_6  \
0   1.513675  -2.815938  -6.550408   3.206314   0.553201  -0.879096   
1  -2.728249   3.317705   1.423394   2.906636   1.455608   2.252142   
2  -2.450139  -1.778951  -3.101248   2.227944   0.006929   0.577677   
3   0.062025   0.406379   2.021522   1.365662  -0.802058  -0.536730   
4   2.307398  -2.700073  -2.877533   2.908809   1.091788   2.537401   

   Feature_7  Feature_8  Feature_9  Feature_10  Is_Fraud  
0   3.421592  -0.231092   2.820313   -1.983360         0  
1   1.885112  -0.633352   3.211897   -3.902439         0  
2  -0.146535  -0.646634  -0.529501    0.890416         0  
3   1.110541   1.132908  -1.947958   -1.328821         0  
4  -1.810111   1.065836   0.921372    2.298057         0  

Phân bố nhãn (0: Bình thường, 1: Gian lận):
Is_Fraud
0    4721
1     279
Name: count, dtype: int64

================ DỰ BÁO KẾT QUẢ ================
Accuracy  (Độ chính xác tổng quan): 0